In [12]:
#!/usr/bin/env python3
"""
conceptnet_query.py
Query local ConceptNet SQLite database (replacement for API-based version).
"""
import sqlite3
import textwrap
from pathlib import Path

# Database location (matches setup script)
DB_FILE = Path.home() / "conceptnet_data" / "conceptnet.db"

RELATIONS = [
    "/r/IsA", "/r/UsedFor", "/r/CapableOf", "/r/AtLocation",
    "/r/HasSubevent", "/r/HasFirstSubevent", "/r/HasLastSubevent",
    "/r/HasPrerequisite", "/r/MotivatedByGoal", "/r/SimilarTo", "/r/ReceivesAction"
]

# Relation labels for display
RELATION_LABELS = {
    "/r/IsA": "is a",
    "/r/UsedFor": "used for",
    "/r/CapableOf": "capable of",
    "/r/AtLocation": "at location",
    "/r/HasSubevent": "has subevent",
    "/r/HasFirstSubevent": "has first subevent",
    "/r/HasLastSubevent": "has last subevent",
    "/r/HasPrerequisite": "has prerequisite",
    "/r/MotivatedByGoal": "motivated by goal",
    "/r/SimilarTo": "similar to",
    "/r/ReceivesAction": "receives action"
}

def concept_uri(term: str) -> str:
    """Create ConceptNet URI for an English term."""
    return f"/c/en/{term.strip().replace(' ', '_')}"

def extract_label(uri: str) -> str:
    """Extract readable label from ConceptNet URI."""
    if not uri:
        return uri
    # /c/en/knife -> knife
    parts = uri.split('/')
    if len(parts) >= 4:
        return parts[-1].replace('_', ' ')
    return uri

def pretty_edge(edge, highlight_uri):
    """Format an edge for display."""
    start_uri = edge['start']
    end_uri = edge['end']
    rel = edge['relation']
    weight = edge['weight']
    surface = edge.get('surfaceText', '')
    
    start_label = extract_label(start_uri)
    end_label = extract_label(end_uri)
    rel_label = RELATION_LABELS.get(rel, rel)
    
    # Determine direction
    if start_uri == highlight_uri:
        direction = "→"
    elif end_uri == highlight_uri:
        direction = "←"
    else:
        direction = "↔"
    
    weight_str = f"w={weight:.2f}" if weight is not None else "w=N/A"
    line = f"- {start_label} \"{rel_label}\" {direction} {end_label}   ({weight_str})"
    
    if surface:
        gloss = textwrap.shorten(surface.replace("[[", "").replace("]]", ""), width=100)
        line += f"\n    e.g., {gloss}"
    
    return line

def query_edges(conn, uri: str, relation: str, limit: int):
    """
    Query edges for a given URI and relation.
    First tries forward direction (uri as start), then falls back to any direction.
    """
    cursor = conn.cursor()
    
    # Try forward direction first
    cursor.execute("""
        SELECT relation, start, end, weight, surfaceText
        FROM edges
        WHERE start = ? AND relation = ?
        ORDER BY weight DESC
        LIMIT ?
    """, (uri, relation, limit))
    
    edges = cursor.fetchall()
    
    if len(edges) >= 1:
        return [
            {
                'relation': row[0],
                'start': row[1],
                'end': row[2],
                'weight': row[3],
                'surfaceText': row[4]
            }
            for row in edges
        ], "forward"
    
    # Fallback to any direction
    cursor.execute("""
        SELECT relation, start, end, weight, surfaceText
        FROM edges
        WHERE (start = ? OR end = ?) AND relation = ?
        ORDER BY weight DESC
        LIMIT ?
    """, (uri, uri, relation, limit))
    
    edges = cursor.fetchall()
    
    return [
        {
            'relation': row[0],
            'start': row[1],
            'end': row[2],
            'weight': row[3],
            'surfaceText': row[4]
        }
        for row in edges
    ], "any"

def main():
    # Check if database exists
    if not DB_FILE.exists():
        print(f"Error: Database not found at {DB_FILE}")
        print("Please run setup_conceptnet.py first to create the database.")
        return
    
    # Interactive prompts
    term = input("Enter a term (e.g., knife, kettle, car): ").strip()
    if not term:
        print("No term provided. Exiting.")
        return
    
    try:
        k = int(input("How many edges per relation? [7]: ").strip() or "7")
    except ValueError:
        k = 7
    
    uri = concept_uri(term)
    print(f"\n=== ConceptNet samples for '{term}' -> {uri} (English) ===\n")
    
    # Connect to database
    conn = sqlite3.connect(DB_FILE)
    
    try:
        for rel in RELATIONS:
            edges, mode = query_edges(conn, uri, rel, k)
            tag = f"(top {k} edges; {'forward' if mode == 'forward' else 'fallback: any direction'})"
            print(f"{rel} {tag}")
            
            if not edges:
                print("  (no edges found)")
            else:
                for edge in edges:
                    print(" ", pretty_edge(edge, uri))
            print()
    
    finally:
        conn.close()

if __name__ == "__main__":
    main()


Enter a term (e.g., knife, kettle, car): knife
How many edges per relation? [7]: 7

=== ConceptNet samples for 'knife' -> /c/en/knife (English) ===

/r/IsA (top 7 edges; forward)
  - knife "is a" → n   (w=N/A)
  - knife "is a" → speciality of thiers in france   (w=N/A)
  - knife "is a" → tool   (w=N/A)

/r/UsedFor (top 7 edges; forward)
  - knife "used for" → alter length of string or rope   (w=N/A)
  - knife "used for" → attaching to rifle   (w=N/A)
  - knife "used for" → attacking enemy   (w=N/A)
  - knife "used for" → boning   (w=N/A)
  - knife "used for" → breaking   (w=N/A)
  - knife "used for" → breaking down into pieces   (w=N/A)
  - knife "used for" → butchering   (w=N/A)

/r/CapableOf (top 7 edges; forward)
  - knife "capable of" → butter bread   (w=N/A)
  - knife "capable of" → butter bred   (w=N/A)
  - knife "capable of" → butter piece of toast   (w=N/A)
  - knife "capable of" → cut   (w=N/A)
  - knife "capable of" → cut apple   (w=N/A)
  - knife "capable of" → cut cake   (w

In [2]:
#!/usr/bin/env python3
"""
setup_conceptnet.py
Downloads ConceptNet 5.7 assertions and builds a SQLite database with indexes.
Run this once to set up your local ConceptNet database.
"""
import os
import sqlite3
import csv
import urllib.request
import gzip
import shutil
from pathlib import Path

# Configuration
DATA_DIR = Path.home() / "conceptnet_data"
CSV_GZ_URL = "https://s3.amazonaws.com/conceptnet/downloads/2019/edges/conceptnet-assertions-5.7.0.csv.gz"
CSV_GZ_FILE = DATA_DIR / "conceptnet-assertions-5.7.0.csv.gz"
CSV_FILE = DATA_DIR / "conceptnet-assertions-5.7.0.csv"
DB_FILE = DATA_DIR / "conceptnet.db"

def download_file(url, dest):
    """Download file with progress indicator."""
    print(f"Downloading {url}...")
    print(f"Destination: {dest}")
    
    def progress(block_num, block_size, total_size):
        downloaded = block_num * block_size
        percent = min(100, downloaded * 100 / total_size)
        print(f"\rProgress: {percent:.1f}% ({downloaded / 1024 / 1024:.1f} MB / {total_size / 1024 / 1024:.1f} MB)", end="")
    
    urllib.request.urlretrieve(url, dest, reporthook=progress)
    print("\nDownload complete!")

def decompress_gzip(gz_file, output_file):
    """Decompress .gz file."""
    print(f"Decompressing {gz_file.name}...")
    with gzip.open(gz_file, 'rb') as f_in:
        with open(output_file, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f"Decompressed to {output_file}")

def create_database(csv_file, db_file):
    """Create SQLite database from CSV file."""
    print(f"Creating SQLite database: {db_file}")
    
    # Remove existing database
    if db_file.exists():
        print("Removing existing database...")
        db_file.unlink()
    
    # Connect to database
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    
    # Create table
    print("Creating table schema...")
    cursor.execute("""
        CREATE TABLE edges (
            relation TEXT NOT NULL,
            start TEXT NOT NULL,
            end TEXT NOT NULL,
            weight REAL,
            dataset TEXT,
            license TEXT,
            sources TEXT,
            surfaceText TEXT
        )
    """)
    
    # Import CSV data
    print("Importing CSV data (this may take several minutes)...")
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')
        
        # Skip header if present
        header = next(reader, None)
        if header and header[0].startswith('URI'):
            print("Skipping header row...")
        else:
            # If no header, process first row
            if header:
                try:
                    weight = float(header[4]) if header[4] else None
                except (ValueError, IndexError):
                    weight = None
                    
                cursor.execute("INSERT INTO edges VALUES (?, ?, ?, ?, ?, ?, ?, ?)", 
                             (header[1], header[2], header[3], 
                              weight,
                              header[5] if len(header) > 5 else None,
                              header[6] if len(header) > 6 else None,
                              header[7] if len(header) > 7 else None,
                              header[8] if len(header) > 8 else None))
        
        # Process remaining rows
        count = 0
        batch = []
        batch_size = 10000
        
        for row in reader:
            if len(row) < 4:
                continue
                
            # Parse weight safely
            try:
                weight = float(row[4]) if row[4] else None
            except (ValueError, IndexError):
                weight = None
            
            batch.append((
                row[1],  # relation
                row[2],  # start
                row[3],  # end
                weight,  # weight
                row[5] if len(row) > 5 else None,  # dataset
                row[6] if len(row) > 6 else None,  # license
                row[7] if len(row) > 7 else None,  # sources
                row[8] if len(row) > 8 else None   # surfaceText
            ))
            
            count += 1
            
            if len(batch) >= batch_size:
                cursor.executemany("INSERT INTO edges VALUES (?, ?, ?, ?, ?, ?, ?, ?)", batch)
                batch = []
                if count % 100000 == 0:
                    print(f"  Imported {count:,} rows...")
        
        # Insert remaining rows
        if batch:
            cursor.executemany("INSERT INTO edges VALUES (?, ?, ?, ?, ?, ?, ?, ?)", batch)
        
        print(f"  Total rows imported: {count:,}")
    
    conn.commit()
    
    # Create indexes
    print("Creating indexes (this may take a few minutes)...")
    print("  Creating index on 'start' column...")
    cursor.execute("CREATE INDEX idx_start ON edges(start)")
    
    print("  Creating index on 'end' column...")
    cursor.execute("CREATE INDEX idx_end ON edges(end)")
    
    print("  Creating index on 'relation' column...")
    cursor.execute("CREATE INDEX idx_relation ON edges(relation)")
    
    print("  Creating composite index on (start, relation)...")
    cursor.execute("CREATE INDEX idx_start_relation ON edges(start, relation)")
    
    conn.commit()
    
    # Statistics
    cursor.execute("SELECT COUNT(*) FROM edges")
    total = cursor.fetchone()[0]
    print(f"\nDatabase created successfully!")
    print(f"  Total edges: {total:,}")
    print(f"  Database file: {db_file}")
    print(f"  Database size: {db_file.stat().st_size / 1024 / 1024:.1f} MB")
    
    conn.close()

def main():
    print("=== ConceptNet Local Setup ===\n")
    
    # Create data directory
    DATA_DIR.mkdir(exist_ok=True)
    print(f"Data directory: {DATA_DIR}\n")
    
    # Step 1: Download
    if not CSV_GZ_FILE.exists():
        download_file(CSV_GZ_URL, CSV_GZ_FILE)
    else:
        print(f"Compressed file already exists: {CSV_GZ_FILE}")
    
    # Step 2: Decompress
    if not CSV_FILE.exists():
        decompress_gzip(CSV_GZ_FILE, CSV_FILE)
    else:
        print(f"CSV file already exists: {CSV_FILE}")
    
    # Step 3: Create database
    if not DB_FILE.exists():
        create_database(CSV_FILE, DB_FILE)
    else:
        print(f"\nDatabase already exists: {DB_FILE}")
        response = input("Rebuild database? (y/N): ").strip().lower()
        if response == 'y':
            create_database(CSV_FILE, DB_FILE)
        else:
            print("Keeping existing database.")
    
    print("\n=== Setup Complete ===")
    print(f"You can now use conceptnet_query.py to query the database.")
    print(f"Database location: {DB_FILE}")
    
    # Optional: Clean up CSV files to save space
    print(f"\nOptional: You can delete the CSV files to save ~2GB of disk space:")
    print(f"  rm {CSV_FILE}")
    print(f"  rm {CSV_GZ_FILE}")

if __name__ == "__main__":
    main()


=== ConceptNet Local Setup ===

Data directory: /home/sheema/conceptnet_data

Destination: /home/sheema/conceptnet_data/conceptnet-assertions-5.7.0.csv.gz
Progress: 100.0% (474.9 MB / 474.9 MB)
Download complete!
Decompressing conceptnet-assertions-5.7.0.csv.gz...
Decompressed to /home/sheema/conceptnet_data/conceptnet-assertions-5.7.0.csv
Creating SQLite database: /home/sheema/conceptnet_data/conceptnet.db
Creating table schema...
Importing CSV data (this may take several minutes)...
  Imported 100,000 rows...
  Imported 200,000 rows...
  Imported 300,000 rows...
  Imported 400,000 rows...
  Imported 500,000 rows...
  Imported 600,000 rows...
  Imported 700,000 rows...
  Imported 800,000 rows...
  Imported 900,000 rows...
  Imported 1,000,000 rows...
  Imported 1,100,000 rows...
  Imported 1,200,000 rows...
  Imported 1,300,000 rows...
  Imported 1,400,000 rows...
  Imported 1,500,000 rows...
  Imported 1,600,000 rows...
  Imported 1,700,000 rows...
  Imported 1,800,000 rows...
  Impo

  Imported 25,900,000 rows...
  Imported 26,000,000 rows...
  Imported 26,100,000 rows...
  Imported 26,200,000 rows...
  Imported 26,300,000 rows...
  Imported 26,400,000 rows...
  Imported 26,500,000 rows...
  Imported 26,600,000 rows...
  Imported 26,700,000 rows...
  Imported 26,800,000 rows...
  Imported 26,900,000 rows...
  Imported 27,000,000 rows...
  Imported 27,100,000 rows...
  Imported 27,200,000 rows...
  Imported 27,300,000 rows...
  Imported 27,400,000 rows...
  Imported 27,500,000 rows...
  Imported 27,600,000 rows...
  Imported 27,700,000 rows...
  Imported 27,800,000 rows...
  Imported 27,900,000 rows...
  Imported 28,000,000 rows...
  Imported 28,100,000 rows...
  Imported 28,200,000 rows...
  Imported 28,300,000 rows...
  Imported 28,400,000 rows...
  Imported 28,500,000 rows...
  Imported 28,600,000 rows...
  Imported 28,700,000 rows...
  Imported 28,800,000 rows...
  Imported 28,900,000 rows...
  Imported 29,000,000 rows...
  Imported 29,100,000 rows...
  Imported